# Neuropixel Spike Sorting with SpikeInterface

This notebook walks through preprocessing and spike sorting for data recorded with a **Neuropixel probe** using the [SpikeInterface](https://spikeinterface.readthedocs.io/) framework.

**Assumed data format:** SpikeGLX (`.bin` / `.meta` files)

---

## 1. Imports & Job Configuration

Standard scientific Python libraries plus the SpikeInterface modules used throughout this notebook.

A few things worth noting:
- **`OPENBLAS_NUM_THREADS`** is set to `1` to prevent OpenBLAS from spawning its own thread pool, which would conflict with SpikeInterface's parallel job system and cause CPU oversubscription.
- **`SLURM_CPUS_PER_TASK`** is read from the environment so the same notebook runs correctly both locally and on a SLURM cluster. If the variable is not set (i.e. you're running locally), it defaults to `8`.
- SpikeInterface's **global job kwargs** control parallelism for all compute-heavy steps (`n_jobs`, `chunk_duration`, `progress_bar`). Setting them once here means you don't need to pass them to every function call.

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import shutil
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

%matplotlib inline

# Prevent OpenBLAS from spawning its own threads — SpikeInterface manages
# parallelism through its own job system, so this avoids CPU oversubscription.
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# ---------------------------------------------------------------------------
# Parallelism configuration
# ---------------------------------------------------------------------------
# On a SLURM cluster, SLURM_CPUS_PER_TASK is set automatically by the
# scheduler. Locally it won't exist, so we fall back to 8.
# We reserve one CPU for the main process and hand the rest to workers.
from spikeinterface.core import set_global_job_kwargs

cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", "8"))
global_job_kwargs = dict(
    n_jobs=cpus - 1,       # number of parallel worker processes
    chunk_duration="5s",   # length of data chunks processed per worker
    progress_bar=True,     # show tqdm progress bars for long-running steps
)
set_global_job_kwargs(**global_job_kwargs)

print(f"Using {cpus - 1} worker(s) ({cpus} CPUs available)")

# ---------------------------------------------------------------------------
# SpikeInterface imports
# ---------------------------------------------------------------------------
from spikeinterface.extractors import get_neo_streams, read_spikeglx
from spikeinterface.core import get_noise_levels, create_sorting_analyzer
import spikeinterface.widgets as sw
from spikeinterface.preprocessing import apply_preprocessing_pipeline, bandpass_filter
from spikeinterface.sortingcomponents.peak_detection import detect_peaks
from spikeinterface.sortingcomponents.peak_localization import localize_peaks
from spikeinterface.core.motion import Motion
from spikeinterface.sortingcomponents.motion import estimate_motion, correct_motion_on_peaks, InterpolateMotionRecording
from spikeinterface.sorters import get_default_sorter_params, run_sorter
from spikeinterface.metrics import compute_quality_metrics

# ---------------------------------------------------------------------------
# ProbeInterface imports
# ---------------------------------------------------------------------------
from probeinterface.plotting import plot_probe

## 2. Load Raw Recording

Point to the SpikeGLX data directory and load the AP-band stream for the desired probe (imec index).

- **`DATA_PATH`** — directory containing the `.bin` and `.meta` files for one probe
- **`IMEC`** — probe index (0-based); change this if you have multiple probes
- **`load_sync_channel=False`** — excludes the sync/TTL channel from the electrode array so it doesn't interfere with preprocessing or sorting

In [ ]:
# ---------------------------------------------------------------------------
# Paths & parameters — edit these for your dataset
# ---------------------------------------------------------------------------
DATA_PATH = '/ix1/pmayo/lab_NHPdata/kendra_scrappy_0136a_g0/kendra_scrappy_0136a_g0_imec0'
IMEC = 0  # probe index; basically which imec slot did you plug into on the chassis

# ---------------------------------------------------------------------------
# Inspect available streams, then load the AP-band recording
# ---------------------------------------------------------------------------
# get_neo_streams() lists all streams in the SpikeGLX folder (AP, LFP, etc.)
stream_names, stream_ids = get_neo_streams('spikeglx', DATA_PATH)
print("Available streams:", stream_names)

# Load the action-potential (30 kHz) band; sync channel excluded
raw_recording = read_spikeglx(DATA_PATH, stream_name=f'imec{IMEC}.ap', load_sync_channel=False)

# ---------------------------------------------------------------------------
# [OPTIONAL] Trim to first 10 minutes for a quick test run
# ---------------------------------------------------------------------------
# Uncomment the line below to work with a short segment before committing
# to processing the full recording. Frames are in samples.
#raw_recording = raw_recording.frame_slice(start_frame=0, end_frame=int(10*60*raw_recording.get_sampling_frequency()))

# ---------------------------------------------------------------------------
# Sanity check — confirm the recording looks as expected before proceeding
# ---------------------------------------------------------------------------
duration_min = round((raw_recording.get_num_samples(segment_index=0) /
                      raw_recording.get_sampling_frequency()) / 60)

print("\n---------------- RAW RECORDING --------------------")
print("Sampling frequency (Hz): ", raw_recording.get_sampling_frequency())
print("Number of channels:      ", raw_recording.get_num_channels())
print("Number of segments:      ", raw_recording.get_num_segments())
print("Number of samples:       ", raw_recording.get_num_samples(segment_index=0))
print("Duration (min):          ", duration_min)
print("Data dtype:              ", raw_recording.get_dtype())
print("---------------------------------------------------")

### Noise Level Check

A quick sanity check before sorting: plot the per-channel noise distribution in both raw (`int16`) and scaled (`µV`) units. Channels with unusually high noise that survived bad-channel detection can be removed manually if needed.

In [ ]:
def plot_noise_levels(recording):
    """Histogram of per-channel noise in both µV and raw int16 units."""
    noise_uV    = get_noise_levels(recording, return_in_uV=True)
    noise_int16 = get_noise_levels(recording, return_in_uV=False)

    fig, axs = plt.subplots(ncols=2, figsize=(10, 5), squeeze=False, sharey=True)

    axs[0, 0].hist(noise_uV,    bins=np.arange(5,  30, 2.5), color='gray', edgecolor='black')
    axs[0, 0].set_xlabel('Noise (µV)')

    axs[0, 1].hist(noise_int16, bins=np.arange(0,  15, 2.5), color='gray', edgecolor='black')
    axs[0, 1].set_xlabel('Noise (int16)')

    fig.text(0, 0.5, '# of channels', va='center', rotation='vertical', fontsize=10)
    fig.suptitle('Per-channel noise levels (raw recording)', fontsize=12)
    plt.tight_layout()
    plt.show()

    return noise_int16

# Plot noise levels
noise_levels_int16 = plot_noise_levels(raw_recording)

### Probe Map

Verify that SpikeInterface correctly read the probe geometry from the SpikeGLX `.meta` file. The table shows each channel's coordinates and the plot shows their physical layout on the shank.

In [ ]:
# Inspect channel coordinates as a table
raw_recording.get_probe().to_dataframe()

# Plot the physical layout of electrodes on the shank
# with_channel_ids=True labels each site with its channel ID
fig, ax = plt.subplots(figsize=(15, 10))
plot_probe(raw_recording.get_probe(), ax=ax)
ax.set_ylim(-100, 500)
plt.show()

## 3. Preprocessing

Raw Neuropixel data requires several preprocessing steps before spike sorting. The pipeline is defined as an ordered dictionary — steps are applied sequentially in the order listed.

| Step | What it does |
|---|---|
| `phase_shift` | Corrects for the time offset between channels introduced by the Neuropixel's multiplexed ADC sampling |
| `highpass_filter` | Removes low-frequency LFP/drift by high-pass filtering at 300 Hz; output is cast to `int16` to save memory |
| `detect_and_remove_bad_channels` | Identifies and removes noisy or dead channels automatically, this step is optional |
| `common_reference` | Subtracts the median across all channels at each timepoint to suppress common-mode noise |

In [ ]:
# ---------------------------------------------------------------------------
# Preprocessing pipeline definition
# ---------------------------------------------------------------------------
# Steps are applied in order. Adjust parameters here as needed.
PREPROCESSING_DICT = {
    "phase_shift": {},                          # correct ADC sampling offsets
    "highpass_filter": {
        "freq_min": 300,                        # high-pass cutoff (Hz)
        "dtype": "int16"                        # cast output to save memory
    },
    "detect_and_remove_bad_channels": {},       # auto-detect and drop bad channels
    "common_reference": {
        "operator": "median",                   # subtract median across channels
        "reference": "global"                   # use all channels to compute reference
    },
}

In [ ]:
def plot_preprocessing_steps(raw_recording, preprocessing_dict):
    """Plot a trace map after each preprocessing step to visually verify the pipeline."""
    pp_steps = list(preprocessing_dict.items())

    # One subplot per step, plus one for the raw signal
    fig, axs = plt.subplots(
        ncols=len(pp_steps) + 1,
        figsize=(30, 10),
        squeeze=False,
        sharex=True,
        sharey=True,
    )

    # Raw traces
    sw.plot_traces(raw_recording, backend='matplotlib', clim=(-50, 50), ax=axs[0, 0], mode="map")
    axs[0, 0].set_title("raw")

    # Apply steps one at a time and plot after each
    recording = raw_recording
    for i, (step_name, step_params) in enumerate(pp_steps, start=1):
        print(f"Step {i}/{len(pp_steps)}: '{step_name}'  |  params: {step_params}")
        recording = apply_preprocessing_pipeline(recording, dict(pp_steps[:i]))
        sw.plot_traces(recording, backend='matplotlib', clim=(-50, 50), ax=axs[0, i], mode="map")
        axs[0, i].set_title(step_name)

    fig.text(0.5, 0, 'Time (seconds)', ha='center', fontsize=14)
    fig.text(0, 0.5, 'Channels', va='center', rotation='vertical', fontsize=14)
    plt.tight_layout()
    plt.show()


# Visualize what each step does to the raw traces
#plot_preprocessing_steps(raw_recording, PREPROCESSING_DICT)

# Apply preprocessing pipeline to raw recording
pp_recording = apply_preprocessing_pipeline(raw_recording, PREPROCESSING_DICT)

print("\n------------- PREPROCESS RECORDING --------------")
print("Number of channels:      ", pp_recording.get_num_channels())
print("---------------------------------------------------")

## 4. Motion Correction

Rather than using the high-level `correct_motion()` wrapper, we run each step manually so we can inspect intermediate results and tune parameters independently:

1. **Bandpass filter** — broader frequency range (300–5000 Hz) gives cleaner peaks for localization
2. **Detect peaks** — find candidate spike events across the probe
3. **Localize peaks** — estimate the spatial (x, y) position of each peak
4. **Visualize drift** — scatter plot of peak positions over time to confirm drift is present and sensible
5. **Estimate motion** — fit a motion model to the localized peaks
6. **Apply correction** — interpolate `pp_recording` (not the bandpass version) using the estimated motion

In [ ]:
# Bandpass filter for motion estimation only — NOT used for sorting
rec_bandpass = bandpass_filter(recording=pp_recording, freq_min=300., freq_max=5000.)

MOTION_KWARGS = {
    "detect_kwargs": {},
    "localize_peaks_kwargs": {},
    "estimate_motion_kwargs": {
        "win_scale_um": 1000.0,   # spatial smoothing window (µm)
        "motion_bound": 800,      # max expected drift (µm)
        "time_kernel_width": 30,  # temporal smoothing kernel width (s)
        "method": "medicine",
    },
    "interpolate_motion_kwargs": {
        "border_mode": "force_extrapolate"
    },
}

In [ ]:
# Detect candidate spike events across all channels
peaks = detect_peaks(
    rec_bandpass,
    noise_levels=noise_levels_int16,
    method="locally_exclusive",
    **MOTION_KWARGS['detect_kwargs'],
)

# Estimate the (x, y) position of each peak on the probe
peak_locations = localize_peaks(
    rec_bandpass,
    peaks,
    method="monopolar_triangulation",
    **MOTION_KWARGS['localize_peaks_kwargs'],
)

print(f"Detected {len(peaks):,} peaks")

In [ ]:
print(peaks)

fs = rec_bandpass.sampling_frequency

# Peak y-position over time — drift shows up as slow vertical trends
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(peaks['sample_index'] / fs, peak_locations['y'], color='k', marker='.', alpha=0.002)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Depth (µm)')
ax.set_title('Peak locations over time')
plt.show()

# Peak x/y positions overlaid on the probe geometry
fig, ax = plt.subplots(figsize=(15, 10))
plot_probe(rec_bandpass.get_probe(), ax=ax)
ax.scatter(peak_locations['x'], peak_locations['y'], color='purple', alpha=0.002)
ax.set_ylim(-100, 500)
ax.set_title('Peak locations on probe')
plt.show()

In [ ]:
# Plot motion summary: drift traces, peak locations, and corrected raster
sw.plot_motion_info(motion_info, recording=pp_recording)
plt.show()

In [ ]:
# Fit the motion model to the localized peaks
motion_output = estimate_motion(
    recording=rec_bandpass,
    peaks=peaks,
    peak_locations=peak_locations,
    **MOTION_KWARGS['estimate_motion_kwargs'],
)

In [ ]:
# Summary plot: drift traces
motion_object = Motion(
            displacement=[motion_output.displacement[0]],
            temporal_bins_s=[motion_output.temporal_bins_s[0]],
            spatial_bins_um=motion_output.spatial_bins_um,
        )

fig = plt.figure(figsize=(14, 8))
sw.plot_motion(
    motion_object
)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Apply motion correction to the sorting-ready recording
# ---------------------------------------------------------------------------
# Interpolation is applied to pp_recording (not rec_bandpass) so the final
# recording has the correct preprocessing chain for spike sorting.
# astype(float) is required by InterpolateMotionRecording.
mc_recording = InterpolateMotionRecording(
    pp_recording.astype(float),
    motion_object,
    **MOTION_KWARGS['interpolate_motion_kwargs'],
)

## 5. Spike Sorting (Kilosort 4)

Spike sorting is run via SpikeInterface's `run_sorter()` wrapper around [Kilosort4](https://github.com/MouseLand/Kilosort).

A few important notes:
- **`do_correction=False`** — KS4's built-in drift correction is disabled because we already applied motion correction in the previous step. Running both would double-correct.
- The input recording is `mc_recording` — the motion-corrected, preprocessed signal.
- Output is saved to `kilosort4_output/` inside `DATA_PATH`. Set `remove_existing_folder=True` to overwrite a previous run, or `False` to reload it.

### Parameter Choices

Parameters left at their defaults are not listed here. The table below summarises what was changed and why.

| Parameter | Default | Ours | Reason |
|---|---|---|---|
| `do_correction` | `True` | `False` | Motion correction already applied; disables KS4's internal drift correction |
| `dminx` | `32` | `103` | Horizontal template spacing (µm); increased to match the lateral contact spacing of this probe geometry |
| `max_channel_distance` | `32` | `120` | Max distance (µm) between a template and its nearest channel; increased to capture wider-field units |
| `whitening_range` | `32` | `17` | Number of nearby channels used to compute the whitening matrix; reduced to keep whitening local |
| `nearest_chans` | `10` | `17` | Channels considered when finding local spike maxima; increased for denser search |
| `Th_universal` | `9` | `8` | Detection threshold for universal templates (in whitened SD units); slightly lowered to catch more spikes |
| `Th_learned` | `8` | `7` | Detection threshold for learned templates; slightly lowered to match |
| `ccg_threshold` | `0.25` | `0.25` | CCG refractory violation fraction allowed for splits/merges; kept at default |
| `acg_threshold` | `0.2` | `0.2` | ACG refractory violation fraction for labelling a unit "good"; tightened for stricter quality |

In [ ]:
get_default_sorter_params('kilosort4')

In [ ]:
params_kilosort4 = {
    "sorter_name": "kilosort4",

    # --- drift correction ---
    "do_correction": False,         # disabled — motion correction already applied above

    # --- spike detection ---
    "dminx": 103,                    # horizontal template spacing (µm); set to match probe lateral pitch
    "max_channel_distance": 120,    # max template-to-channel distance (µm); increased from default 32
    "nearest_chans": 17,            # channels searched for local maxima; increased from default 10
    "Th_universal": 8,              # universal template detection threshold (default 9)
    "Th_learned": 7,                # learned template detection threshold (default 8)

    # --- preprocessing ---
    "whitening_range": 17,          # channels used to estimate whitening matrix; reduced from default 32

    # --- clustering / postprocessing ---
    "ccg_threshold": 0.25,          # CCG refractory violation tolerance for merges (default 0.25)
    "acg_threshold": 0.2,           # ACG refractory violation tolerance for "good" label (default 0.2)
}

In [ ]:
sorting = run_sorter(
    recording=mc_recording,
    folder=Path(DATA_PATH) / 'kilosort4_output',
    verbose=True,
    save_preprocessed_copy=False,
    docker_image=False,
    clear_cache=True,
    remove_existing_folder=True,    # set False to reload an existing run
    **params_kilosort4,
)

print(f"\nSorting complete: {len(sorting.unit_ids)} units found")

## 6. Postprocessing

After sorting, we compute a set of quality metrics and waveform features using SpikeInterface's `SortingAnalyzer`. The analyzer stores the sorting result alongside the recording and computes **extensions** — modular analyses that build on each other.

Extensions are computed in dependency order automatically, but it's worth knowing how they chain:
- `random_spikes` → selects a subset of spikes used by `waveforms`
- `waveforms` → required by `templates`, `principal_components`, `spike_amplitudes`, `amplitude_scalings`, `spike_locations`
- `templates` → required by `template_metrics`, `template_similarity`, `unit_locations`

The table below describes each extension we compute:

| Extension | What it computes |
|---|---|
| `isi_histograms` | Inter-spike interval distributions per unit |
| `correlograms` | Cross- and auto-correlograms between unit pairs |
| `acgs_3d` | 3D ACGs (time × amplitude × ISI) for richer refractory period assessment |
| `noise_levels` | Per-channel noise estimate used to scale amplitudes |
| `random_spikes` | Random subsample of spikes per unit used for waveform extraction |
| `waveforms` | Raw waveform snippets around each spike (±1.5 / 2 ms) |
| `principal_components` | Per-channel PCA of waveforms (3 components, local channels only) |
| `templates` | Mean, median, and std waveform templates per unit |
| `amplitude_scalings` | Per-spike amplitude scaling factors relative to the template |
| `spike_amplitudes` | Absolute amplitude of each spike on its best channel |
| `spike_locations` | Spatial position of each spike estimated from template geometry |
| `template_metrics` | Single- and multi-channel waveform shape features (width, PTR, spread, etc.) |
| `template_similarity` | Pairwise template cosine similarity matrix — useful for catching over-splits |
| `unit_locations` | Estimated centre-of-mass position of each unit on the probe |

In [ ]:
EXTENSIONS = {
    # --- quality metrics ---
    "isi_histograms": {},
    "correlograms": {},
    "acgs_3d": {},
    "noise_levels": {},

    # --- waveform extraction (random_spikes must come first) ---
    "random_spikes": {
        "method": "uniform",
        "max_spikes_per_unit": 500,  # cap per unit to keep memory manageable
        "seed": 69,
    },
    "waveforms": {
        "ms_before": 1.5,            # snippet window before spike peak (ms)
        "ms_after": 2,               # snippet window after spike peak (ms)
    },
    "principal_components": {
        "n_components": 3,           # PCs per channel
        "mode": "by_channel_local",  # fit PCA independently on each channel
    },
    "templates": {
        "operators": ["average", "median", "std"],
    },

    # --- amplitude & location (require waveforms + templates) ---
    "amplitude_scalings": {},
    "spike_amplitudes": {},
    "spike_locations": {},

    # --- template-level features ---
    "template_metrics": {
        "include_multi_channel_metrics": True,  # adds spread, velocity, etc.
    },
    "template_similarity": {},
    "unit_locations": {},
}

In [ ]:
# Create the SortingAnalyzer — links the sorting result to the recording
# and sets up the folder where all extensions will be saved.
#   sparse=True       : only store waveforms on the subset of channels nearest
#                       each unit (saves substantial disk space)
#   return_in_uV=True : scale all outputs to microvolts
analyzer = create_sorting_analyzer(
    sorting=sorting,
    recording=mc_recording,
    format="binary_folder",
    sparse=True,
    return_in_uV=True,
    folder=Path(DATA_PATH) / 'analyzer',
)

print(analyzer)

In [ ]:
# Compute all extensions in one call.
# Extensions are resolved in dependency order automatically.
# This is the most compute-intensive step — expect it to take several minutes.
analyzer.compute(EXTENSIONS, **global_job_kwargs)

print("\nPostprocessing complete.")
print(analyzer)

## 7. Quality Metrics

Quality metrics summarise how well-isolated and physiologically plausible each unit is. They are computed from the analyzer extensions (waveforms, amplitudes, spike trains, etc.) and saved to a CSV for downstream curation.

Metrics are grouped loosely by what they assess:

**Spike train statistics**
| Metric | What it measures |
|---|---|
| `num_spikes` | Total spike count per unit |
| `firing_rate` | Mean firing rate (Hz) over the recording |
| `presence_ratio` | Fraction of recording epochs in which the unit fires — low values suggest instability |
| `firing_range` | Difference between max and min firing rate across epochs — captures rate non-stationarity |
| `synchrony` | Tendency of a unit to fire synchronously with other units |

**Refractory period / contamination**
| Metric | What it measures |
|---|---|
| `isi_violation` | Fraction of ISIs below the refractory period (2 ms); proxy for false positive rate |
| `rp_violation` | Refractory period violation rate using the Hill et al. estimator |
| `sliding_rp_violation` | Sliding-window version of `rp_violation`; more robust to non-stationarity |

**Amplitude-based**
| Metric | What it measures |
|---|---|
| `snr` | Peak template amplitude divided by channel noise |
| `amplitude_cutoff` | Estimates the fraction of spikes missing from the low end of the amplitude distribution (proxy for false negative rate) |
| `amplitude_median` | Median spike amplitude (µV) |
| `amplitude_cv` | Coefficient of variation of spike amplitudes — high values may indicate multi-unit activity |
| `noise_cutoff` | Checks whether the amplitude distribution has an unnatural cutoff near the noise floor |
| `sd_ratio` | Ratio of spike amplitude SD to noise SD |

**Drift**
| Metric | What it measures |
|---|---|
| `drift` | Estimated vertical displacement of the unit over the recording |

**Cluster separation**
| Metric | What it measures |
|---|---|
| `nearest_neighbor` | Fraction of nearest neighbours in PCA space that belong to the same unit |
| `mahalanobis` | Mahalanobis-distance-based isolation from other units in feature space |
| `d_prime` | Linear discriminability between a unit and its nearest neighbour |

In [ ]:
QUALITY_METRICS = {
    # --- spike train ---
    "num_spikes": {},
    "firing_rate": {},
    "presence_ratio": {},
    "firing_range": {},
    "synchrony": {},

    # --- refractory period / contamination ---
    "isi_violation": {},
    "rp_violation": {},
    "sliding_rp_violation": {},

    # --- amplitude ---
    "snr": {},
    "amplitude_cutoff": {},
    "amplitude_median": {},
    "amplitude_cv": {},
    "noise_cutoff": {},
    "sd_ratio": {},

    # --- drift ---
    "drift": {},

    # --- cluster separation ---
    "nearest_neighbor": {},
    "mahalanobis": {},
    "d_prime": {},
}

In [ ]:
metrics = compute_quality_metrics(
    analyzer,
    metric_names=list(QUALITY_METRICS.keys()),
    metric_params=QUALITY_METRICS,
)

# Save to CSV for use in downstream curation / plotting
metrics.to_csv(Path(DATA_PATH) / 'cluster_metrics.csv', index=True)

print(f"Computed metrics for {len(metrics)} units.")
metrics.head()

## 8. Visualization

Two levels of visualization for inspecting sorting results:

1. **Session summary** — a single figure that gives a bird's-eye view of all units in a session: depth distribution, amplitudes, presence over time, template similarity, and key quality metric distributions.
2. **Unit summary** — a per-unit figure for inspecting individual clusters in detail (waveforms, ACG, amplitude over time, PC projections).

### Session Summary

`plot_analyzer_sess()` lays out nine panels in a single figure:

| Panel | Content |
|---|---|
| Unit depths | Probe depth vs. unit index — shows spatial distribution of sorted units |
| Amplitude distributions | Per-unit amplitude spread across all spikes |
| Unit presence | Firing activity heatmap over the session — identifies unstable or transient units |
| Template similarity | Pairwise cosine similarity matrix — high off-diagonal values flag potential over-splits |
| Firing rate | Distribution of mean firing rates across units |
| Presence ratio | Fraction of epochs in which each unit is active |
| SNR | Signal-to-noise ratio distribution |
| RP contamination | Refractory period contamination distribution |
| Amplitude cutoff | Estimated fraction of missing spikes per unit |

In [ ]:
def plot_analyzer_sess(analyzer, metrics):
    """
    Session-level summary figure for a sorted recording.

    Parameters
    ----------
    analyzer : SortingAnalyzer
    metrics  : pd.DataFrame  — output of compute_quality_metrics()
    profile  : optional metadata (unused; reserved for future annotation)
    """
    fig = plt.figure(figsize=(12, 12))
    gs = GridSpec(10, 5, figure=fig)

    ax0 = fig.add_subplot(gs[0:9, 0])        # unit depths
    ax1 = fig.add_subplot(gs[0:2, 1:4])      # amplitude distributions
    ax2 = fig.add_subplot(gs[2:4, 1:4])      # unit presence
    ax3 = fig.add_subplot(gs[4:7, 1:3])      # template similarity
    ax4 = fig.add_subplot(gs[4:6, 3:4])      # firing rate
    ax5 = fig.add_subplot(gs[6:7, 3:4])      # presence ratio
    ax6 = fig.add_subplot(gs[7:9, 1:2])      # SNR
    ax7 = fig.add_subplot(gs[7:9, 2:3])      # RP contamination
    ax8 = fig.add_subplot(gs[7:9, 3:4])      # amplitude cutoff

    # --- probe depth layout ---
    sw.plot_unit_depths(analyzer, ax=ax0)

    # --- amplitude distributions (one per unit) ---
    sw.plot_all_amplitudes_distributions(analyzer, ax=ax1)
    ax1.set_xlabel("clusters")
    ax1.set_xticks([])
    ax1.set_ylabel("amplitude")

    # --- unit presence over time ---
    session_duration_min = (analyzer.recording.get_num_frames() / analyzer.sampling_frequency) / 60
    sw.plot_unit_presence(analyzer, time_range=[0, session_duration_min], ax=ax2)
    ax2.set_ylabel("clusters")
    ax2.figure.axes[-1].set_ylabel("normalized activity")

    # --- template similarity matrix ---
    sw.plot_template_similarity(analyzer, ax=ax3)
    ax3.set_xlabel("cluster 1")
    ax3.set_ylabel("cluster 2")
    ax3.figure.axes[-1].set_ylabel("template similarity")

    # --- quality metric histograms ---
    def _hist(ax, col, xlabel, ylabel=None):
        mean_val = metrics[col].mean()
        ax.hist(metrics[col], bins=20)
        ax.axvline(mean_val, linestyle="--", color="red")
        ax.set_xlabel(xlabel)
        ax.set_title(f"mean = {mean_val:.2f}")
        if ylabel:
            ax.set_ylabel(ylabel)

    _hist(ax4, "firing_rate",      "firing rate (Hz)",   ylabel="# clusters")
    _hist(ax5, "presence_ratio",   "presence ratio",     ylabel="# clusters")
    _hist(ax6, "snr",              "SNR",                ylabel="# clusters")
    _hist(ax7, "rp_contamination", "rp contamination")
    _hist(ax8, "amplitude_cutoff", "amplitude cutoff")

    plt.tight_layout()
    plt.show()


# Make summary plot
plot_analyzer_sess(analyzer, metrics)

### Unit Summary

Drill into any individual unit by its ID. The `UnitSummaryWidget` shows the mean waveform on nearby channels, ACG, spike amplitude over time, and PCA projections — the standard set of panels for manual curation decisions.

In [ ]:
# Change cluster_id to inspect a different unit
cluster_id = analyzer.unit_ids[0]

fig = plt.figure(figsize=(12, 10))
UnitSummaryWidget(analyzer, unit_id=cluster_id, backend="matplotlib", figure=fig)
plt.show()